# 🌱 Soil Analyzer — v3 · Dashboard Interactivo
**Autor:** Ariel Macías | Agrónomo · GIS & Remote Sensing  

Este notebook analiza suelos agrícolas usando datos oficiales USDA-NRCS (Soil Data Access)  
y presenta un **dashboard de dos bloques** con promedio ponderado en zona de raíces y perfiles en profundidad.

| Celda | Descripción |
|-------|-------------|
| 0 | Configuración |
| 1 | Inicialización GEE y definición del lote |
| 2 | Descarga WFS de polígonos de suelo |
| 3 | **Nueva** — Consulta chorizon completa (todos los horizontes) |
| 4 | **Nueva** — Función de promedio ponderado 0–80 cm |
| 5 | **Bloque 1** — Mapa coropleta (variable seleccionable) |
| 6 | **Bloque 2** — Perfiles en profundidad con Plotly |

### Variables consultadas (tabla `chorizon`)
| Variable | Columna SDA | Nota |
|----------|-------------|------|
| Arena | `sandtotal_r` | |
| Limo | `silttotal_r` | |
| Arcilla | `claytotal_r` | |
| pH | `ph1to1h2o_r` | |
| CE (Salinidad) | `ec_r` | |
| PSI (Sodicidad) | `esp_r` | |
| SAR | `sar_r` | |
| Materia Orgánica | `om_r` | |
| CIC | `cec7_r` | |
| **Fósforo (P)** | — | ⚠️ No disponible en SDA |
| **Nitrógeno (N)** | — | Estimado desde MO (N = MO / 20) |

> ⚠️ Los datos USDA-NRCS tienen cobertura en **Estados Unidos**.

In [1]:
# ============================================================
# CELDA 0 — CONFIGURACIÓN
# ============================================================

GEE_PROJECT  = 'my-project-12126-484118'   # <-- Reemplazá con tu project ID de GEE
CENTRO_LAT   = 33.584              # Latitud del centro del mapa inicial
CENTRO_LON   = -101.845            # Longitud del centro del mapa inicial
ZOOM_INICIAL = 14

# Profundidad máxima de la zona de raíces (cm)
PROF_MAX_CM  = 80

# Factor de conversión MO → Nitrógeno orgánico total estimado
# Basado en la relación C:N de Bremner (1965): N ≈ MO / 20
FACTOR_N     = 1 / 20

print("✅ Configuración cargada.")
print(f"   Proyecto GEE    : {'my-project-12126-484118'}")
print(f"   Centro mapa     : ({CENTRO_LAT}, {CENTRO_LON})")
print(f"   Profundidad raíz: 0–{PROF_MAX_CM} cm")
print(f"   Factor N/MO     : 1/{int(1/FACTOR_N)}  (N estimado = MO / {int(1/FACTOR_N)})")

✅ Configuración cargada.
   Proyecto GEE    : my-project-12126-484118
   Centro mapa     : (33.584, -101.845)
   Profundidad raíz: 0–80 cm
   Factor N/MO     : 1/20  (N estimado = MO / 20)


---
## Paso 1 · Dibujar o cargar el lote
Usá la herramienta de polígono del panel izquierdo, o subí un Shapefile/GeoJSON.

In [ ]:
# ============================================================
# CELDA 1 — INICIALIZACIÓN GEE Y DIBUJO / CARGA DEL LOTE
# ============================================================
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import io
import urllib
import warnings
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import folium
from folium import GeoJson, Choropleth
from folium.features import GeoJsonTooltip
import branca.colormap as cm
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union
from IPython.display import display, HTML, FileLink
from contextlib import redirect_stdout
import json
import tempfile
import os

warnings.filterwarnings('ignore')

# --- Autenticación GEE ---
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine inicializado.")
except Exception:
    print("🔑 Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine inicializado.")

# Variables compartidas
lote_geom     = None   # ee.Geometry
lote_geom_shp = None   # shapely geometry

# ============================================================
# MAPA INTERACTIVO
# ============================================================
Draw_Map = geemap.Map(center=[CENTRO_LAT, CENTRO_LON], zoom=ZOOM_INICIAL)
Draw_Map.add_basemap('SATELLITE')

btn_confirmar = widgets.Button(
    description='✅ Confirmar polígono dibujado',
    button_style='success',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_dibujo = widgets.Output()

def confirmar_poligono(b):
    global lote_geom, lote_geom_shp
    with out_dibujo:
        out_dibujo.clear_output()
        roi = Draw_Map.user_roi
        if roi is None:
            print("⚠️ No se detectó ningún polígono. Dibujá uno primero.")
            return
        roi_info  = roi.getInfo()
        lote_geom = ee.Geometry(roi_info.get('geometry', roi_info))
        coords    = lote_geom.coordinates().getInfo()[0]
        lote_geom_shp = Polygon(coords)
        print(f"✅ Polígono confirmado — {len(coords)-1} vértices")
        print(f"   Área aprox.: {lote_geom_shp.area * 111320**2 / 10000:.2f} ha")
        local_path = "lote_exportado.geojson"
        geojson_str = json.dumps({"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": mapping(lote_geom_shp), "properties": {"nombre": "lote"}}]}, indent=2)
        with open(local_path, "w") as f: f.write(geojson_str)
        print("📥 Descargá el polígono:"); display(FileLink(local_path, result_html_prefix="👉 "))

btn_confirmar.on_click(confirmar_poligono)

# ── Carga de Shapefile / GeoJSON ──
uploader = widgets.FileUpload(description='📂 Subir Lote (.shp / .geojson)', accept='.shp,.zip,.geojson', multiple=True, layout=widgets.Layout(width='260px', margin='4px'))
btn_cargar = widgets.Button(description='📌 Cargar SHP al mapa', button_style='info', layout=widgets.Layout(width='260px', margin='4px'))
out_shp = widgets.Output()

def cargar_shp(b):
    global lote_geom, lote_geom_shp
    with out_shp:
        out_shp.clear_output()
        if not uploader.value:
            print("⚠️ No se subió ningún archivo."); return
        tmp_dir = tempfile.mkdtemp()
        for archivo in uploader.value:
            with open(os.path.join(tmp_dir, archivo['name']), 'wb') as f: f.write(archivo['content'])
        try:
            geojson_files = [f for f in os.listdir(tmp_dir) if f.endswith('.geojson')]
            shp_files     = [f for f in os.listdir(tmp_dir) if f.endswith('.shp')]
            zip_files     = [f for f in os.listdir(tmp_dir) if f.endswith('.zip')]
            if zip_files:
                import zipfile
                with zipfile.ZipFile(os.path.join(tmp_dir, zip_files[0]), 'r') as z: z.extractall(tmp_dir)
                shp_files = [f for f in os.listdir(tmp_dir) if f.endswith('.shp')]
            archivo_path = os.path.join(tmp_dir, (geojson_files or shp_files)[0]) if (geojson_files or shp_files) else None
            if not archivo_path: print("❌ No se encontró .shp ni .geojson."); return
            gdf = gpd.read_file(archivo_path).to_crs(epsg=4326)
            lote_geom_shp = unary_union(gdf.geometry)
            lote_geom     = ee.Geometry(mapping(lote_geom_shp))
            print(f"✅ Archivo cargado — Área: {lote_geom_shp.area * 111320**2 / 10000:.2f} ha")
            with redirect_stdout(io.StringIO()):
                Draw_Map.add_gdf(gdf, layer_name="Lote Vectorial", style={'color': '#f1c40f', 'weight': 2, 'fillOpacity': 0.2})
            print("🗺️  Lote agregado al mapa.")
        except Exception as e:
            print(f"❌ Error: {e}")

btn_cargar.on_click(cargar_shp)

titulo = widgets.HTML("<h3 style='margin:8px 0'>📍 Definición del lote</h3>")
panel_dibujo = widgets.VBox([widgets.HTML("<b>Opción A — Dibujar en el mapa</b>"), btn_confirmar, out_dibujo], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='4px', width='300px'))
panel_shp    = widgets.VBox([widgets.HTML("<b>Opción B — Subir Shapefile / GeoJSON</b>"), uploader, btn_cargar, out_shp], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='4px', width='300px'))

display(titulo)
display(Draw_Map)
display(widgets.HBox([panel_dibujo, panel_shp]))

✅ Earth Engine inicializado.


HTML(value="<h3 style='margin:8px 0'>📍 Definición del lote</h3>")

Map(center=[33.584, -101.845], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

---
## Paso 2 · Descarga WFS de polígonos de suelo

In [4]:
# ============================================================
# CELDA 2 — DESCARGA WFS Y RECORTE AL LOTE
# ============================================================

if lote_geom is None:
    if Draw_Map.user_roi is not None:
        roi_coords    = Draw_Map.user_roi.getInfo()['coordinates'][0]
        lote_geom     = ee.Geometry.Polygon([roi_coords])
        lote_geom_shp = Polygon(roi_coords)
    else:
        raise ValueError("⚠️ No se definió ningún lote. Volvé a Celda 1.")

shapely_poly = lote_geom_shp
centroid     = lote_geom.centroid(maxError=1).coordinates().getInfo()
centro_lote  = [centroid[1], centroid[0]]
minx, miny, maxx, maxy = shapely_poly.bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"

wfs_url = "https://sdmdataaccess.nrcs.usda.gov/Spatial/SDMNAD83Geographic.wfs"
params  = {'SERVICE': 'WFS', 'VERSION': '1.0.0', 'REQUEST': 'GetFeature',
           'TYPENAME': 'MapunitPoly', 'BBOX': bbox_str,
           'SRSNAME': 'EPSG:4326', 'OUTPUTFORMAT': 'GML3'}

print("📡 Conectando al Soil Data Mart USDA (WFS)...")
response = requests.get(wfs_url, params=params, verify=False, timeout=30)

if response.status_code != 200:
    raise ConnectionError(f"❌ Error al conectar con USDA: {response.status_code}")

gdf_soils = gpd.read_file(io.BytesIO(response.content))
if gdf_soils.empty:
    raise ValueError("⚠️ Sin datos de suelo. Recordá que USDA-NRCS solo cubre EE.UU.")

gdf_lote    = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[shapely_poly])
gdf_clipped = gpd.overlay(gdf_soils, gdf_lote, how='intersection')
gdf_clipped['hectareas'] = gdf_clipped.to_crs(epsg=3857).area / 10000

mukeys_list = gdf_clipped['mukey'].unique().tolist()
mukeys_str  = f"({mukeys_list[0]})" if len(mukeys_list) == 1 else str(tuple(mukeys_list)).replace(',)', ')')

print(f"✅ {len(gdf_clipped)} polígonos descargados y recortados.")
print(f"   Unidades cartográficas: {len(mukeys_list)}")
print(f"   MUKeys: {mukeys_list}")

display

📡 Conectando al Soil Data Mart USDA (WFS)...
✅ 25 polígonos descargados y recortados.
   Unidades cartográficas: 13
   MUKeys: [94072, 94093, 3390163, 94054, 94055, 94046, 94059, 94083, 94062, 94042, 94039, 94091, 94100]


<function IPython.core.display_functions.display(*objs, include=None, exclude=None, metadata=None, transient=None, display_id=None, raw=False, clear=False, **kwargs)>

In [5]:
# ========================================
# 3. INFORME DE SUELOS (LINKS MODERNOS SDE + TABULAR SDA)
# ========================================
import requests, pandas as pd, urllib3, urllib.parse
from IPython.display import display, HTML

# Silenciamos advertencias de SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("\n--- PASO 3: GENERANDO REPORTE DE SUELOS (SDA + UC DAVIS) ---")

# 1. Preparar MUKEYs y calcular Hectáreas (Asegurando que la columna existe)
gdf_temp = gdf_clipped.copy()
gdf_temp['hectareas'] = gdf_temp.to_crs(epsg=3857).area / 10000

resumen_mukey = gdf_temp.groupby(['mukey', 'musym']).agg({'hectareas': 'sum'}).reset_index()
resumen_mukey = resumen_mukey.sort_values(by='hectareas', ascending=False)
total_ha = resumen_mukey['hectareas'].sum()
muke_list = resumen_mukey['mukey'].unique().tolist()
mukeys_str = str(tuple(muke_list)).replace(",)", ")")

# 2. SQL para traer componentes y texturas (Capa superficial 0-20cm aprox)
sql_query = f"""
SELECT c.mukey, c.compname, c.comppct_r, ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r, ch.om_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str}
AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

# 3. Petición a la base de datos SDA
series_data = pd.DataFrame()
try:
    url_sda = "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest"
    payload = {"query": sql_query, "format": "JSON"}
    r = requests.post(url_sda, data=payload, verify=False, timeout=30)
    if r.status_code == 200:
        json_data = r.json()
        if 'Table' in json_data:
            # Los nombres de columnas deben coincidir con el orden del SELECT
            series_data = pd.DataFrame(json_data['Table'], columns=['mukey', 'serie', 'pct', 'arena', 'limo', 'arcilla', 'mo'])
except Exception as e:
    print(f"❌ Error al conectar con USDA: {e}")

# 4. Construcción del HTML con los LINKS DE UC DAVIS
html_output = f"<div style='font-family: sans-serif; max-width: 950px;'><h2 style='color: #2c3e50; border-bottom: 3px solid #2c3e50;'>📊 Informe de Suelos: Composición y Texturas</h2>"

for _, fila in resumen_mukey.iterrows():
    detalles = series_data[series_data['mukey'].astype(str) == str(fila['mukey'])] if not series_data.empty else pd.DataFrame()
    pct_lote = (fila['hectareas'] / total_ha) * 100

    html_output += f"""
    <details style="margin-bottom: 12px; border: 1px solid #bdc3c7; border-radius: 6px; overflow: hidden;">
        <summary style="background-color: #2c3e50; padding: 15px; font-weight: bold; cursor: pointer; color: #fff;">
            📍 Unidad: {fila['musym']} — {fila['hectareas']:.2f} ha ({pct_lote:.1f}%)
        </summary>
        <div style="padding: 15px; background: #fff;">
            <table style="width: 100%; border-collapse: collapse;">
                <thead>
                    <tr style="background-color: #ecf0f1; color: #2c3e50;">
                        <th style="padding: 10px; text-align: left; border: 1px solid #ddd;">Serie de Suelo</th>
                        <th style="border: 1px solid #ddd;">%</th>
                        <th style="border: 1px solid #ddd;">Arena %</th>
                        <th style="border: 1px solid #ddd;">Limo %</th>
                        <th style="border: 1px solid #ddd;">Arcilla %</th>
                        <th style="border: 1px solid #ddd; color:#e67e22;">M.O. %</th>
                    </tr>
                </thead>
                <tbody>
    """
    for _, s in detalles.iterrows():
        nombre_s = str(s['serie']).strip()
        # Codificación para URL (ej: 'Acuff' -> 'acuff')
        nombre_s_url = urllib.parse.quote(nombre_s.lower())
        url_ficha = f"https://casoilresource.lawr.ucdavis.edu/sde/?series={nombre_s_url}#osd"

        html_output += f"""
                <tr style="border-bottom: 1px solid #eee;">
                    <td style="padding: 10px; font-weight: bold; border: 1px solid #ddd;">
                        <a href="{url_ficha}" target="_blank" style="color: #3498db; text-decoration: none;">{nombre_s}</a> 🔗
                    </td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['pct']}%</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['arena'] if s['arena'] else '-'}</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['limo'] if s['limo'] else '-'}</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #333; ">{s['arcilla'] if s['arcilla'] else '-'}</td>
                    <td style="text-align: center; border: 1px solid #ddd; color: #d35400; font-weight: bold;">{s['mo'] if s['mo'] else '-'}</td>
                </tr>"""
    html_output += "</tbody></table></div></details>"

display(HTML(html_output + "</div>"))

# ============================================================
# CELDA 2 — DESCARGA WFS Y MAPA COLOREADO POR TEXTURA
# ============================================================

# 1. Verificación — acepta polígono dibujado O shapefile cargado
if lote_geom is None:
    if Draw_Map.user_roi is not None:
        # Fallback: si confirmaron sin usar el botón (raro, pero cubrimos)
        roi_coords   = Draw_Map.user_roi.getInfo()['coordinates'][0]
        lote_geom    = ee.Geometry.Polygon([roi_coords])
        lote_geom_shp = Polygon(roi_coords)
    else:
        raise ValueError("⚠️ No se definió ningún lote. Volvé a la Celda 1, dibujá o subí un SHP y confirmá.")

# 2. Geometría del AOI — usa lote_geom_shp sin importar el origen
shapely_poly = lote_geom_shp
centroid     = lote_geom.centroid(maxError=1).coordinates().getInfo()
centro_lote  = [centroid[1], centroid[0]]   # [lat, lon]

minx, miny, maxx, maxy = shapely_poly.bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"

# 3. Consulta WFS a USDA Soil Data Mart
wfs_url = "https://sdmdataaccess.nrcs.usda.gov/Spatial/SDMNAD83Geographic.wfs"
params  = {
    'SERVICE'     : 'WFS',
    'VERSION'     : '1.0.0',
    'REQUEST'     : 'GetFeature',
    'TYPENAME'    : 'MapunitPoly',
    'BBOX'        : bbox_str,
    'SRSNAME'     : 'EPSG:4326',
    'OUTPUTFORMAT': 'GML3'
}

print("📡 Conectando al Soil Data Mart USDA (WFS)...")
response = requests.get(wfs_url, params=params, verify=False, timeout=30)

if response.status_code != 200:
    raise ConnectionError(f"❌ Error al conectar con USDA: {response.text}")

gdf_soils = gpd.read_file(io.BytesIO(response.content))

if gdf_soils.empty:
    raise ValueError("⚠️ No se encontraron datos de suelo en esta área. Recordá que USDA-NRCS solo tiene cobertura en EE.UU.")

gdf_lote    = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[shapely_poly])
gdf_clipped = gpd.overlay(gdf_soils, gdf_lote, how='intersection')
print(f"✅ {len(gdf_clipped)} unidades de suelo descargadas y recortadas al lote.")

# 4. Consulta tabular para textura dominante
mukeys_list = gdf_clipped['mukey'].unique().tolist()
mukeys_str  = str(tuple(mukeys_list)).replace(',)', ')')
if len(mukeys_list) == 1:
    mukeys_str = f"({mukeys_list[0]})"

sql_textura = f"""
SELECT c.mukey, c.compname, c.comppct_r,
       ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str}
  AND c.majcompflag = 'Yes'
  AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

def clasificar_textura(sand, silt, clay):
    if any(v is None for v in [sand, silt, clay]):
        return 'Sin datos'
    s, si, c = float(sand), float(silt), float(clay)
    if c >= 40 and s <= 45 and si <= 40: return 'Arcilloso'
    if c >= 40 and si >= 40:             return 'Arcillo limoso'
    if c >= 35 and s >= 45:              return 'Arcillo arenoso'
    if c >= 27 and s <= 45 and si >= 28: return 'Franco arcilloso'
    if c >= 27 and s >= 45:              return 'Franco arcillo arenoso'
    if c >= 27 and si >= 60:             return 'Franco arcillo limoso'
    if si >= 50 and c < 27:              return 'Franco limoso'
    if si >= 80 and c < 12:              return 'Limoso'
    if s >= 85 and c < 10:               return 'Arenoso'
    if s >= 70 and c < 15:               return 'Franco arenoso'
    return 'Franco'

PALETA_TEXTURA = {
    'Arcilloso'             : '#8B0000',
    'Arcillo limoso'        : '#B22222',
    'Arcillo arenoso'       : '#CD5C5C',
    'Franco arcilloso'      : '#D2691E',
    'Franco arcillo arenoso': '#A0522D',
    'Franco arcillo limoso' : '#C68642',
    'Franco limoso'         : '#DEB887',
    'Limoso'                : '#F5DEB3',
    'Franco'                : '#8FBC8F',
    'Franco arenoso'        : '#F4A460',
    'Arenoso'               : '#EDC9Af',
    'Sin datos'             : '#AAAAAA',
}

df_tex = pd.DataFrame()
try:
    r_tex = requests.post(
        "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
        data={'query': sql_textura, 'format': 'JSON'},
        verify=False, timeout=30
    )
    if r_tex.status_code == 200 and 'Table' in r_tex.json():
        df_tex = pd.DataFrame(
            r_tex.json()['Table'],
            columns=['mukey','compname','comppct_r','sand','silt','clay']
        )
        df_tex['textura'] = df_tex.apply(
            lambda row: clasificar_textura(row['sand'], row['silt'], row['clay']), axis=1
        )
except Exception as e:
    print(f"⚠️ No se pudo obtener textura para colorear el mapa: {e}")

# 5. Mapa coloreado por clase textural
mapa_suelos = geemap.Map(center=centro_lote, zoom=14)
mapa_suelos.add_basemap('SATELLITE')

gdf_map = gdf_clipped[['mukey', 'musym', 'geometry']].copy()

if not df_tex.empty:
    tex_dom = df_tex.sort_values('comppct_r', ascending=False).drop_duplicates('mukey')[['mukey','textura']]
    # ✅ Fix mukey dtype mismatch
    gdf_map['mukey'] = gdf_map['mukey'].astype(str)
    tex_dom['mukey'] = tex_dom['mukey'].astype(str)
    gdf_map = gdf_map.merge(tex_dom, on='mukey', how='left')
    gdf_map['textura'] = gdf_map['textura'].fillna('Sin datos')
else:
    gdf_map['textura'] = 'Sin datos'

for textura, grupo in gdf_map.groupby('textura'):
    color = PALETA_TEXTURA.get(textura, '#AAAAAA')
    style = {'color': color, 'weight': 1, 'fillColor': color, 'fillOpacity': 0.55}
    hover = {'fillOpacity': 0.85}
    with redirect_stdout(io.StringIO()):
        mapa_suelos.add_gdf(grupo, layer_name=f"Textura: {textura}",
                            style=style, hover_style=hover)

with redirect_stdout(io.StringIO()):
    mapa_suelos.add_gdf(gdf_lote, layer_name='Límite del lote',
                        style={'color': 'white', 'weight': 2, 'fillOpacity': 0})

print("✅ Mapa generado. Las capas se pueden alternar desde el panel de capas.")
display(mapa_suelos)




--- PASO 3: GENERANDO REPORTE DE SUELOS (SDA + UC DAVIS) ---


Serie de Suelo,%,Arena %,Limo %,Arcilla %,M.O. %
Heldt 🔗,80%,22.1,27.9,50,1.25
Serie de Suelo,%,Arena %,Limo %,Arcilla %,M.O. %
Olnest 🔗,10%,66.4,19.1,14.5,1.5
Vona 🔗,5%,83.8,9.2,7,1.25
Ascalon 🔗,85%,66.8,19.2,14,1
Serie de Suelo,%,Arena %,Limo %,Arcilla %,M.O. %
Arvada 🔗,80%,41.6,37.4,21,1.25
Serie de Suelo,%,Arena %,Limo %,Arcilla %,M.O. %
Vona 🔗,10%,78,16,6,1
Ascalon 🔗,10%,84,11,5,1.25


📡 Conectando al Soil Data Mart USDA (WFS)...
✅ 25 unidades de suelo descargadas y recortadas al lote.
✅ Mapa generado. Las capas se pueden alternar desde el panel de capas.


Map(center=[39.85884855326483, -104.16860350000366], controls=(WidgetControl(options=['position', 'transparent…

---
## Paso 3+4 · Horizontes por MUKey + Promedios ponderados 0–80 cm

Cada unidad cartográfica (**MUKey**) se presenta en un bloque expandible con:
- Horizontes del **componente dominante**
- **Promedio ponderado 0–80 cm** (fila con fondo azul tenue)

**Atributos físicos:** Textura, Densidad aparente, Contenido de agua (10, 33, 1500 kPa)  
**Atributos químicos:** MO, pH (H₂O 1:1 + pasta saturada), CE, ECEC, PSI, SAR, CIC, Ca, K, N, S, P

**Sistema de alertas (paleta suave):**
- 🔴 **CRÍTICO** — acción urgente recomendada  
- 🟡 **ALERTA** — monitoreo y diagnóstico adicional  
- 🟢 **ÓPTIMO** — condición favorable

> Variables con alerta activa: CE (salinidad), PSI y SAR (sodicidad), pH (acidez/alcalinidad), Arcilla % (compactación/infiltración), MO, Densidad aparente (compactación física).

In [6]:
# ============================================================
# CELDA 3+4 — CONSULTA COMPLETA DE HORIZONTES + PROMEDIOS PONDERADOS
# Tabla interactiva por MUKey con alertas de valores críticos
# ============================================================
import math

# ── SQL principal: todos los horizontes + propiedades físicas y químicas ────
sql_horizontes = (
    "SELECT\n"
    "    c.mukey, c.cokey, c.compname, c.comppct_r, c.majcompflag,\n"
    "    ch.chkey, ch.hzname, ch.hzdept_r, ch.hzdepb_r,\n"
    "    ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r,\n"
    "    ch.dbovendry_r,\n"
    "    ch.wtenthbar_r, ch.wthirdbar_r, ch.wfifteenbar_r,\n"
    "    ch.om_r,\n"
    "    ch.ph1to1h2o_r,\n"
    "    ch.ec_r, ch.ecec_r,\n"
    "    ch.esp_r, ch.sar_r,\n"
    "    ch.cec7_r\n"
    "FROM component AS c\n"
    "LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey\n"
    f"WHERE c.mukey IN {mukeys_str}\n"
    "  AND ch.hzdept_r IS NOT NULL\n"
    "  AND ch.hzdepb_r IS NOT NULL\n"
    "ORDER BY c.mukey, c.comppct_r DESC, ch.hzdept_r ASC"
)

COLS_HORIZ = [
    'mukey', 'cokey', 'compname', 'comppct_r', 'majcompflag',
    'chkey', 'hzname', 'hzdept_r', 'hzdepb_r',
    'sandtotal_r', 'silttotal_r', 'claytotal_r',
    'dbovendry_r',
    'wtenthbar_r', 'wthirdbar_r', 'wfifteenbar_r',
    'om_r',
    'ph1to1h2o_r',
    'ec_r', 'ecec_r',
    'esp_r', 'sar_r',
    'cec7_r'
]

COLS_NUM = [
    'comppct_r', 'hzdept_r', 'hzdepb_r',
    'sandtotal_r', 'silttotal_r', 'claytotal_r',
    'dbovendry_r',
    'wtenthbar_r', 'wthirdbar_r', 'wfifteenbar_r',
    'om_r',
    'ph1to1h2o_r',
    'ec_r', 'ecec_r',
    'esp_r', 'sar_r',
    'cec7_r'
]

df_horiz = pd.DataFrame()
try:
    print("📡 Consultando tabla chorizon (todos los horizontes + propiedades físicas/químicas)...")
    r = requests.post(
        "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
        data={'query': sql_horizontes, 'format': 'JSON'},
        verify=False, timeout=60
    )
    if r.status_code == 200 and 'Table' in r.json():
        df_horiz = pd.DataFrame(r.json()['Table'], columns=COLS_HORIZ)
        for col in COLS_NUM:
            df_horiz[col] = pd.to_numeric(df_horiz[col], errors='coerce')
        df_horiz['n_estimado_r'] = df_horiz['om_r'] * FACTOR_N
        df_horiz['espesor_cm']   = df_horiz['hzdepb_r'] - df_horiz['hzdept_r']
        print(f"✅ {len(df_horiz)} horizontes descargados para {df_horiz['mukey'].nunique()} MUKeys.")
    else:
        print(f"⚠️ El servidor SDA devolvió código: {r.status_code}")
        print(r.text[:500])
except Exception as e:
    print(f"❌ Error al consultar SDA: {e}")


# ── Funciones auxiliares ────────────────────────────────────

def promedio_ponderado_zona_raiz(df_comp, variable, prof_max=PROF_MAX_CM):
    """Promedio ponderado por espesor efectivo en zona de raíces (0–prof_max cm)."""
    df = df_comp.copy()
    df['top_eff'] = df['hzdept_r'].clip(lower=0, upper=prof_max)
    df['bot_eff'] = df['hzdepb_r'].clip(lower=0, upper=prof_max)
    df['esp_eff'] = df['bot_eff'] - df['top_eff']
    df = df[(df['esp_eff'] > 0) & df[variable].notna()].copy()
    if df.empty: return np.nan
    total_esp = df['esp_eff'].sum()
    return (df[variable] * df['esp_eff']).sum() / total_esp if total_esp > 0 else np.nan


def calcular_wpavg_por_mukey(df_horiz, variable, prof_max=PROF_MAX_CM):
    """Calcula wpavg por mukey usando el componente dominante."""
    resultados = []
    for mukey, df_mu in df_horiz.groupby('mukey'):
        comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
        df_comp  = df_mu[df_mu['cokey'] == comp_dom]
        valor    = promedio_ponderado_zona_raiz(df_comp, variable, prof_max)
        resultados.append({'mukey': str(mukey), f'{variable}_wpavg': valor})
    return pd.DataFrame(resultados)


def clasificar_textura(sand, silt, clay):
    if any(v is None or (isinstance(v, float) and np.isnan(v)) for v in [sand, silt, clay]):
        return 'Sin datos'
    s, si, c = float(sand), float(silt), float(clay)
    if c >= 40 and s <= 45 and si <= 40: return 'Arcilloso'
    if c >= 40 and si >= 40:             return 'Arcillo limoso'
    if c >= 35 and s >= 45:              return 'Arcillo arenoso'
    if c >= 27 and s <= 45 and si >= 28: return 'Franco arcilloso'
    if c >= 27 and s >= 45:              return 'Franco arcillo arenoso'
    if c >= 27 and si >= 60:             return 'Franco arcillo limoso'
    if si >= 50 and c < 27:              return 'Franco limoso'
    if si >= 80 and c < 12:              return 'Limoso'
    if s >= 85 and c < 10:               return 'Arenoso'
    if s >= 70 and c < 15:               return 'Franco arenoso'
    return 'Franco'


PALETA_TEXTURA = {
    'Arcilloso'             : '#8B0000',
    'Arcillo limoso'        : '#B22222',
    'Arcillo arenoso'       : '#CD5C5C',
    'Franco arcilloso'      : '#D2691E',
    'Franco arcillo arenoso': '#A0522D',
    'Franco arcillo limoso' : '#C68642',
    'Franco limoso'         : '#DEB887',
    'Limoso'                : '#F5DEB3',
    'Franco'                : '#8FBC8F',
    'Franco arenoso'        : '#F4A460',
    'Arenoso'               : '#EDC9Af',
    'Sin datos'             : '#AAAAAA',
}

VARIABLES_MAP = {
    'Arcilla (%)':            'claytotal_r',
    'Arena (%)':              'sandtotal_r',
    'Limo (%)':               'silttotal_r',
    'Densidad aparente (g/cm³)': 'dbovendry_r',
    'Agua 10 kPa (%)':        'wtenthbar_r',
    'Agua 33 kPa (%)':        'wthirdbar_r',
    'Agua 1500 kPa (%)':      'wfifteenbar_r',
    'Materia Orgánica (%)':   'om_r',
    'pH H₂O 1:1':             'ph1to1h2o_r',
    'CE — Salinidad (dS/m)':  'ec_r',
    'ECEC (cmol/kg)':         'ecec_r',
    'PSI — Sodicidad (%)':    'esp_r',
    'SAR':                    'sar_r',
    'CIC (cmol/kg)':          'cec7_r',
    'N estimado (%) ⚠️':      'n_estimado_r',
}

RENAME_WPAVG = {
    'claytotal_r_wpavg'   : 'Arcilla % (0–80cm)',
    'sandtotal_r_wpavg'   : 'Arena % (0–80cm)',
    'silttotal_r_wpavg'   : 'Limo % (0–80cm)',
    'dbovendry_r_wpavg'   : 'Dens. ap. (0–80cm)',
    'wtenthbar_r_wpavg'   : 'Agua 10kPa (0–80cm)',
    'wthirdbar_r_wpavg'   : 'Agua 33kPa (0–80cm)',
    'wfifteenbar_r_wpavg' : 'Agua 1500kPa (0–80cm)',
    'om_r_wpavg'          : 'MO % (0–80cm)',
    'ph1to1h2o_r_wpavg'   : 'pH H₂O (0–80cm)',
    'ec_r_wpavg'          : 'CE (0–80cm)',
    'ecec_r_wpavg'        : 'ECEC (0–80cm)',
    'esp_r_wpavg'         : 'PSI % (0–80cm)',
    'sar_r_wpavg'         : 'SAR (0–80cm)',
    'cec7_r_wpavg'        : 'CIC (0–80cm)',
    'n_estimado_r_wpavg'  : 'N est. % (0–80cm) ⚠️',
}


# ── Calcular promedios ponderados para todos los MUKeys ────
# Inicialización segura de gdf_dash (fallback si df_horiz está vacío)
gdf_dash = gdf_clipped.copy()
gdf_dash['mukey'] = gdf_dash['mukey'].astype(str)

if not df_horiz.empty:
    wpavg_frames = []
    for label, col in VARIABLES_MAP.items():
        if col in df_horiz.columns:
            wpavg_frames.append(calcular_wpavg_por_mukey(df_horiz, col))
    df_wpavg = wpavg_frames[0].copy()
    for df_tmp in wpavg_frames[1:]:
        df_wpavg = df_wpavg.merge(df_tmp, on='mukey', how='outer')

    # GeoDataFrame enriquecido (usado por Bloques 1 y 2)
    gdf_dash = gdf_clipped.copy()
    gdf_dash['mukey'] = gdf_dash['mukey'].astype(str)
    gdf_dash = gdf_dash.merge(df_wpavg, on='mukey', how='left')

    print(f"✅ Promedios ponderados calculados (0–{PROF_MAX_CM} cm) para {len(df_wpavg)} MUKeys.")


# ── Funciones de alerta visual (softer colors) ─────────────

def celda_alerta(col, val):
    """Devuelve estilos CSS según nivel de alerta — paleta suave."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return 'color:#999; text-align:center;'

    CRITICO = ('background:#ffe4e1; color:#8b0000; font-weight:600; '
               'text-align:center; border-radius:3px;')
    ALERTA  = ('background:#fff9e6; color:#b8860b; font-weight:600; '
               'text-align:center; border-radius:3px;')
    BUENO   = 'background:#f0f9f0; color:#2d6a2d; text-align:center;'
    NORMAL  = 'text-align:center; color:#333;'

    v = float(val)
    if col == 'ec_r':
        return CRITICO if v >= 4 else (ALERTA if v >= 2 else NORMAL)
    if col == 'esp_r':
        return CRITICO if v >= 15 else (ALERTA if v >= 6 else NORMAL)
    if col == 'sar_r':
        return CRITICO if v >= 13 else (ALERTA if v >= 9 else NORMAL)
    if col in ('ph1to1h2o_r', 'phsaturated_r'):
        if v <= 4.5 or v >= 9.0: return CRITICO
        if v <= 5.5 or v >= 8.5: return ALERTA
    if col == 'claytotal_r':
        return CRITICO if v >= 50 else (ALERTA if v >= 40 else NORMAL)
    if col == 'om_r':
        if v < 0.5: return CRITICO
        if v < 1.0: return ALERTA
        if v >= 3.0: return BUENO
    if col in ('cec7_r', 'ecec_r'):
        return CRITICO if v < 6 else (ALERTA if v < 10 else NORMAL)
    if col == 'dbovendry_r':
        return CRITICO if v >= 1.75 else (ALERTA if v >= 1.6 else NORMAL)
    return NORMAL


def fmt(val, decimals=1):
    """Formatea un valor numérico o devuelve '—' si es nulo."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return '<span style="color:#ccc;">—</span>'
    return f"{val:.{decimals}f}"


# ── Definición de columnas de la tabla (formato soft) ──────
COL_DEFS = [
    ('Horizonte',           'hzname',         '',    0),
    ('Prof. (cm)',          '__depth__',       '',    0),
    ('Arena %',             'sandtotal_r',     '',    1),
    ('Limo %',              'silttotal_r',     '',    1),
    ('Arcilla %',           'claytotal_r',     '🏔️',  1),
    ('Clase Textural',      '__textura__',     '',    0),
    ('Dens. Ap. (g/cm³)',   'dbovendry_r',     '',    2),
    ('Agua CC 10kPa (%)',   'wtenthbar_r',     '',    1),
    ('Agua CC 33kPa (%)',   'wthirdbar_r',     '',    1),
    ('Agua PMP 1500kPa (%)', 'wfifteenbar_r',  '',    1),
    ('Mat. Orgánica (%)',   'om_r',            '🌿',  2),
    ('pH (H₂O 1:1)',        'ph1to1h2o_r',     '⚗️',  2),
    ('CE (dS/m)',           'ec_r',            '💧',  2),
    ('ECEC (cmol/kg)',      'ecec_r',          '',    1),
    ('PSI (%)',             'esp_r',           '🧂',  1),
    ('SAR',                 'sar_r',           '🧂',  1),
    ('CIC (cmol/kg)',       'cec7_r',          '',    1),
    ('N estimado (%)',      'n_estimado_r',    '',    3),
]


def build_header():
    ths = []
    for label, col, badge, _ in COL_DEFS:
        txt = f"{badge} {label}".strip()
        ths.append(
            f'<th style="padding:7px 9px; background:#f5f5f5; color:#444; '
            f'border:1px solid #ddd; white-space:nowrap; font-size:11px; '
            f'font-weight:600; text-align:center;">{txt}</th>'
        )
    return '<tr>' + ''.join(ths) + '</tr>'


def build_data_row(hz, bg):
    tds = []
    textura = clasificar_textura(
        hz.get('sandtotal_r'), hz.get('silttotal_r'), hz.get('claytotal_r')
    )
    for label, col, badge, decs in COL_DEFS:
        if col == '__depth__':
            top = hz['hzdept_r']; bot = hz['hzdepb_r']
            top_s = str(int(top)) if pd.notna(top) else '?'
            bot_s = str(int(bot)) if pd.notna(bot) else '?'
            tds.append(
                f'<td style="text-align:center; border:1px solid #e8e8e8; '
                f'padding:5px 7px; white-space:nowrap; color:#555; font-size:11px;">'
                f'{top_s}–{bot_s}</td>'
            )
        elif col == '__textura__':
            color = PALETA_TEXTURA.get(textura, '#aaa')
            tds.append(
                f'<td style="text-align:center; border:1px solid #e8e8e8; padding:5px 7px;">'
                f'<span style="background:{color}18; color:{color}; border:1px solid {color}50; '
                f'padding:2px 6px; border-radius:8px; font-size:10px; white-space:nowrap;">'
                f'{textura}</span></td>'
            )
        elif col == 'hzname':
            val = hz.get(col, '')
            tds.append(
                f'<td style="padding:5px 9px; border:1px solid #e8e8e8; '
                f'font-weight:600; color:#333; font-size:12px;">{val or "—"}</td>'
            )
        else:
            val = hz.get(col)
            style = celda_alerta(col, val) if col else 'text-align:center; color:#333;'
            tds.append(
                f'<td style="padding:5px 7px; border:1px solid #e8e8e8; font-size:11px; {style}">'
                f'{fmt(val, decs)}</td>'
            )
    return f'<tr style="background:{bg};">' + ''.join(tds) + '</tr>'


def build_wpavg_row(wp_dict):
    tds = []
    s_w  = wp_dict.get('sandtotal_r_wpavg')
    si_w = wp_dict.get('silttotal_r_wpavg')
    c_w  = wp_dict.get('claytotal_r_wpavg')
    textura_w = clasificar_textura(s_w, si_w, c_w)

    for label, col, badge, decs in COL_DEFS:
        if col == '__depth__':
            tds.append(
                f'<td style="text-align:center; border:1px solid #c8d9e8; '
                f'padding:6px 7px; font-size:10px; color:#567; font-style:italic;">'
                f'0–{PROF_MAX_CM} cm</td>'
            )
        elif col == '__textura__':
            color = PALETA_TEXTURA.get(textura_w, '#aaa')
            tds.append(
                f'<td style="text-align:center; border:1px solid #c8d9e8; padding:6px 7px;">'
                f'<span style="background:{color}18; color:{color}; border:1px solid {color}50; '
                f'padding:2px 6px; border-radius:8px; font-size:10px;">{textura_w}</span></td>'
            )
        elif col == 'hzname':
            tds.append(
                f'<td style="padding:6px 9px; border:1px solid #c8d9e8; '
                f'font-weight:600; color:#456; font-size:11px; white-space:nowrap;">'
                f'⚖️ Promedio ponderado 0–{PROF_MAX_CM}cm</td>'
            )
        else:
            wp_col = f'{col}_wpavg' if col and not col.startswith('__') else None
            val = wp_dict.get(wp_col) if wp_col else None
            base_style = celda_alerta(col, val) if col else 'text-align:center; color:#333;'
            # Soften alert backgrounds on wpavg row
            base_style = (base_style
                          .replace('background:#ffe4e1', 'background:#ffe8e4')
                          .replace('background:#fff9e6', 'background:#fffae8')
                          .replace('background:#f0f9f0', 'background:#f0fbf0'))
            tds.append(
                f'<td style="padding:6px 7px; border:1px solid #c8d9e8; font-size:11px; {base_style}">'
                f'{fmt(val, decs)}</td>'
            )
    return ('<tr style="background:#e8f1f8; border-top:2px solid #567;">'
            + ''.join(tds) + '</tr>')


# ── Mapa mukey → musym y área ──────────────────────────────
gdf_areas = gdf_clipped.copy()
gdf_areas['mukey'] = gdf_areas['mukey'].astype(str)
gdf_areas['area_ha'] = gdf_areas.geometry.to_crs(epsg=6933).area / 10_000
musym_map = gdf_areas.groupby('mukey')['musym'].first().to_dict()
area_map  = gdf_areas.groupby('mukey')['area_ha'].sum().to_dict()


# ── Leyenda de alertas (soft palette) ──────────────────────
LEYENDA_HTML = (
    '<div style="font-family:sans-serif; font-size:11px; margin:12px 0 18px; '
    'padding:10px 16px; background:#fafafa; border:1px solid #ddd; '
    'border-radius:6px; display:flex; flex-wrap:wrap; gap:8px; align-items:center;">'
    '<b style="color:#333; margin-right:5px;">🔔 Alertas:</b>'
    '<span style="background:#ffe4e1; color:#8b0000; padding:2px 8px; '
    'border-radius:10px; font-weight:600; font-size:10px;">🔴 CRÍTICO</span>'
    '<span style="background:#fff9e6; color:#b8860b; padding:2px 8px; '
    'border-radius:10px; font-weight:600; font-size:10px;">🟡 ALERTA</span>'
    '<span style="background:#f0f9f0; color:#2d6a2d; padding:2px 8px; '
    'border-radius:10px; font-size:10px;">🟢 ÓPTIMO</span>'
    '<span style="color:#666; font-size:10px;">'
    '&nbsp;CE ≥2 alerta / ≥4 crítico (dS/m)'
    '&nbsp;|&nbsp; PSI ≥6 alerta / ≥15 crítico (%)'
    '&nbsp;|&nbsp; SAR ≥9 alerta / ≥13 crítico'
    '&nbsp;|&nbsp; pH &lt;5.5 o &gt;8.5 alerta'
    '&nbsp;|&nbsp; Arcilla ≥40% alerta'
    '&nbsp;|&nbsp; MO &lt;1% alerta / &lt;0.5% crítico'
    '&nbsp;|&nbsp; Dens.ap. ≥1.6 alerta / ≥1.75 crítico'
    '</span></div>'
)


# ── Renderizar tablas por MUKey (soft style) ───────────────
if not df_horiz.empty:
    wpavg_lookup = {}
    if 'df_wpavg' in dir() and not df_wpavg.empty:
        wpavg_lookup = df_wpavg.set_index('mukey').to_dict('index')

    html_all = '<div style="font-family:sans-serif; max-width:100%; overflow-x:auto;">'
    html_all += LEYENDA_HTML

    total_mukeys = df_horiz['mukey'].nunique()
    html_all += (
        f'<p style="color:#555; font-size:12px; margin:0 0 12px;">'
        f'📊 <b>{total_mukeys} unidades cartográficas (MUKeys)</b>. '
        f'Cada sección muestra los horizontes del componente dominante '
        f'+ promedio ponderado 0–{PROF_MAX_CM} cm.</p>'
    )

    for mukey, df_mu in df_horiz.groupby('mukey'):
        mukey_str = str(mukey)
        musym   = musym_map.get(mukey_str, mukey_str)
        area_ha = area_map.get(mukey_str, 0)

        comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
        df_dom   = df_mu[df_mu['cokey'] == comp_dom].sort_values('hzdept_r').reset_index(drop=True)
        compname = df_dom['compname'].iloc[0]
        comppct  = df_dom['comppct_r'].iloc[0]
        n_hz     = len(df_dom)

        wp_row = wpavg_lookup.get(mukey_str, {})

        html_all += (
            f'<details open style="margin-bottom:16px; border:1px solid #d0d0d0; '
            f'border-radius:8px; overflow:hidden; box-shadow:0 1px 3px rgba(0,0,0,0.05);">'
            f'<summary style="background:linear-gradient(135deg,#f7f7f7,#e8e8e8); '
            f'padding:11px 16px; font-weight:600; cursor:pointer; color:#333; '
            f'font-size:13px; list-style:none; display:flex; flex-wrap:wrap; '
            f'align-items:center; gap:10px;">'
            f'<span style="background:#fff; padding:3px 9px; '
            f'border-radius:12px; font-size:12px; border:1px solid #ccc;">MUKey {mukey_str}</span>'
            f'<span>Unidad: <b>{musym}</b></span>'
            f'<span style="opacity:0.5;">|</span>'
            f'<span>{compname} <span style="opacity:0.6; font-weight:normal;">({comppct:.0f}% dom.)</span></span>'
            f'<span style="opacity:0.5;">|</span>'
            f'<span style="opacity:0.8;">{area_ha:.1f} ha &nbsp;· {n_hz} horizontes</span>'
            f'</summary>'
            f'<div style="padding:12px; background:#fff; overflow-x:auto;">'
            f'<table style="width:100%; border-collapse:collapse; font-size:12px;">'
            f'<thead>{build_header()}</thead>'
            f'<tbody>'
        )

        for i, (_, hz) in enumerate(df_dom.iterrows()):
            bg = '#fdfdfd' if i % 2 == 0 else '#ffffff'
            html_all += build_data_row(hz, bg)

        html_all += build_wpavg_row(wp_row)
        html_all += '</tbody></table></div></details>'

    html_all += '</div>'
    display(HTML(html_all))
else:
    print("⚠️ df_horiz está vacío. Revisá la conexión al servidor SDA.")


📡 Consultando tabla chorizon (todos los horizontes + propiedades físicas/químicas)...
✅ 163 horizontes descargados para 13 MUKeys.
✅ Promedios ponderados calculados (0–80 cm) para 13 MUKeys.


Horizonte,Prof. (cm),Arena %,Limo %,🏔️ Arcilla %,Clase Textural,Dens. Ap. (g/cm³),Agua CC 10kPa (%),Agua CC 33kPa (%),Agua PMP 1500kPa (%),🌿 Mat. Orgánica (%),⚗️ pH (H₂O 1:1),💧 CE (dS/m),ECEC (cmol/kg),🧂 PSI (%),🧂 SAR,CIC (cmol/kg),N estimado (%)
A,0–15,78.0,17.0,5.0,Franco arenoso,1.63,21.0,15.5,6.0,1.50,7.00,0.10,—,0.0,0.0,4.8,0.075
Bt1,15–25,67.0,20.0,13.0,Franco,1.65,—,19.4,9.8,0.75,7.00,0.10,—,0.0,0.0,11.1,0.038
Bt2,25–41,67.0,20.0,13.0,Franco,1.65,—,19.4,9.8,0.75,7.00,0.10,—,0.0,0.0,11.1,0.038
C,41–203,83.0,11.0,6.0,Franco arenoso,1.67,19.5,14.0,4.8,0.25,7.00,0.10,—,0.0,0.0,5.3,0.013
⚖️ Promedio ponderado 0–80cm,0–80 cm,76.9,15.1,8.1,Franco arenoso,1.66,19.9,16.0,6.7,0.65,7.00,0.10,—,0.0,0.0,7.1,0.032
Horizonte,Prof. (cm),Arena %,Limo %,🏔️ Arcilla %,Clase Textural,Dens. Ap. (g/cm³),Agua CC 10kPa (%),Agua CC 33kPa (%),Agua PMP 1500kPa (%),🌿 Mat. Orgánica (%),⚗️ pH (H₂O 1:1),💧 CE (dS/m),ECEC (cmol/kg),🧂 PSI (%),🧂 SAR,CIC (cmol/kg),N estimado (%)
H1,0–5,41.6,37.4,21.0,Franco,1.39,—,28.0,12.7,1.25,7.90,1.00,—,—,1.0,12.5,0.062
H2,5–10,67.2,15.3,17.5,Franco,1.50,—,21.0,11.7,1.25,7.90,1.00,—,—,1.0,10.0,0.062
H3,10–38,23.3,29.2,47.5,Arcilloso,1.53,—,31.8,24.1,0.75,8.50,9.00,—,—,23.0,25.0,0.038
H4,38–71,49.7,2.8,47.5,Arcillo arenoso,1.53,—,31.6,23.6,0.25,8.80,12.00,—,—,23.0,32.5,0.013


In [7]:
import pandas as pd

def extraer_datos_laboratorio(pedon_key):
    # Armamos la URL exacta
    url = f"https://ncsslabdatamart.sc.egov.usda.gov/rptExecute.aspx?p={pedon_key}&r=1"
    print(f"📡 Extrayendo datos de: {url}")
    
    try:
        # read_html lee todas las tablas de la página web y devuelve una lista de DataFrames
        tablas = pd.read_html(url)
        print(f"✅ ¡Éxito! Se encontraron {len(tablas)} tablas.")
        
        # En la estructura de la USDA, las tablas suelen estar en este orden:
        # tablas[0] = Datos del Sitio y Clasificación (Pedon)
        # tablas[1] = Granulometría (PSDA) y Textura
        # tablas[4] o [5] = Bases, CIC, pH y Sales (depende del pedón)
        
        # Guardamos la tabla de textura como un archivo CSV directamente
        df_textura = tablas[1]
        nombre_archivo = f"pedon_{pedon_key}_textura.csv"
        df_textura.to_csv(nombre_archivo, index=False)
        print(f"💾 Guardado automáticamente como: {nombre_archivo}")
        
        # Devolvemos todas las tablas para que las puedas explorar en el notebook
        return tablas

    except Exception as e:
        print(f"⚠️ Error al conectar o leer la tabla: {e}")

# Ejecutamos la función con tu ejemplo
todas_las_tablas = extraer_datos_laboratorio(165)

# Para ver la tabla de textura en tu pantalla:
# display(todas_las_tablas[1].head())

📡 Extrayendo datos de: https://ncsslabdatamart.sc.egov.usda.gov/rptExecute.aspx?p=165&r=1
⚠️ Error al conectar o leer la tabla: `Import lxml` failed.  Use pip or conda to install the lxml package.


In [8]:
# ============================================================
# CELDA 4 — (Unificada con Celda 3)
# ============================================================
# Todas las funciones y cálculos están en la Celda 3.
print("ℹ️  Celda 4 unificada con Celda 3 — procesamiento completo disponible.")
n_mukeys = df_horiz['mukey'].nunique() if not df_horiz.empty else 0
n_wpavg  = len(df_wpavg) if 'df_wpavg' in dir() else 0
print(f"   df_horiz  : {len(df_horiz)} horizontes | {n_mukeys} MUKeys")
print(f"   df_wpavg  : {n_wpavg} MUKeys con promedio ponderado")
print(f'   Variables incluidas: Textura, Densidad ap., Agua (10/33/1500 kPa),')
print(f'                        MO, pH (H₂O + pasta), CE, ECEC, PSI, SAR,')
print(f'                        CIC, Ca, K, N, S, P')


ℹ️  Celda 4 unificada con Celda 3 — procesamiento completo disponible.
   df_horiz  : 163 horizontes | 13 MUKeys
   df_wpavg  : 13 MUKeys con promedio ponderado
   Variables incluidas: Textura, Densidad ap., Agua (10/33/1500 kPa),
                        MO, pH (H₂O + pasta), CE, ECEC, PSI, SAR,
                        CIC, Ca, K, N, S, P


---
## 🗺️ BLOQUE 1 — Mapa Coropleta (Promedio Ponderado 0–80 cm)
Seleccioná la variable a visualizar con el menú desplegable.  
Cada polígono de unidad cartográfica se colorea según el valor promedio ponderado en la zona de raíces.

> ⚠️ **Fósforo (P):** No disponible en la base de datos SDA — se indica en el panel como dato no disponible.  
> ⚠️ **Nitrógeno (N):** Estimado matemáticamente a partir de la MO según N = MO / 20.

In [9]:
# ============================================================
# CELDA 5 — BLOQUE 1: MAPA COROPLETA INTERACTIVO (FOLIUM)
# ============================================================

# ── Selector de variable ──────────────────────────────────
OPCIONES_MAP = list(VARIABLES_MAP.keys()) + ['⚠️ Fósforo (P) — No disponible en SDA']

selector_var = widgets.Dropdown(
    options=OPCIONES_MAP,
    value='Arcilla (%)',
    description='Variable:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='380px')
)

btn_generar = widgets.Button(
    description='🗺️ Generar Mapa',
    button_style='primary',
    layout=widgets.Layout(width='160px', margin='4px 8px')
)

out_mapa = widgets.Output()

def generar_mapa(b=None):
    with out_mapa:
        out_mapa.clear_output(wait=True)
        label_sel = selector_var.value

        # ── Caso especial: Fósforo no disponible ──
        if 'Fósforo' in label_sel:
            display(HTML("""
            <div style="background:#fff3cd; border:2px solid #ffc107; border-radius:10px;
                        padding:24px 30px; max-width:680px; font-family:sans-serif; margin:12px 0;">
                <h3 style="color:#856404; margin:0 0 10px;">⚠️ Fósforo (P) — Dato No Disponible en SDA</h3>
                <p style="color:#555; margin:0 0 8px;">
                    La base de datos espacial <strong>USDA Soil Data Access (SDA)</strong> no contiene
                    valores de Fósforo disponible (P) para ningún nivel de horizonte.
                </p>
                <p style="color:#555; margin:0 0 8px;">
                    El Fósforo es un nutriente cuya disponibilidad depende de la mineralogía específica
                    y del historial de fertilización, y por ello no se incluye en los modelos
                    pedológicos generalizados de NRCS.
                </p>
                <p style="color:#856404; font-weight:bold; margin:0;">
                    📋 Para obtener datos de P: realizá análisis de suelo con laboratorio certificado
                    o cargá tus propios datos manualmente en el notebook.
                </p>
            </div>
            """))
            return

        col_data = VARIABLES_MAP[label_sel]
        col_wpavg = f'{col_data}_wpavg'

        if col_wpavg not in gdf_dash.columns:
            print(f"⚠️ Columna '{col_wpavg}' no encontrada. Verificá la Celda 4.")
            return

        gdf_plot = gdf_dash[['mukey', 'musym', 'geometry', col_wpavg]].copy()
        gdf_plot = gdf_plot.dropna(subset=[col_wpavg])

        if gdf_plot.empty:
            display(HTML("<div style='padding:16px; background:#f8d7da; border-radius:8px; color:#721c24;'>"
                         "⚠️ No hay datos disponibles para esta variable en los horizontes del lote.</div>"))
            return

        vmin = gdf_plot[col_wpavg].min()
        vmax = gdf_plot[col_wpavg].max()

        # ── Crear mapa Folium (Google Satellite como fondo) ──
        m = folium.Map(
            location=centro_lote,
            zoom_start=14,
            tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
            attr='Google Satellite'
        )
        folium.TileLayer('CartoDB positron', name='Mapa claro').add_to(m)

        # Colormap
        if col_data == 'ph1to1h2o_r':
            colormap = cm.LinearColormap(['#e74c3c','#e67e22','#27ae60','#f39c12','#c0392b'],
                                          vmin=vmin, vmax=vmax, caption=label_sel)
        elif col_data in ('ec_r', 'esp_r', 'sar_r'):
            colormap = cm.LinearColormap(['#27ae60','#f39c12','#e74c3c'], vmin=vmin, vmax=vmax, caption=label_sel)
        else:
            colormap = cm.LinearColormap(['#fffde7','#f9a825','#e65100'], vmin=vmin, vmax=vmax, caption=label_sel)

        colormap.add_to(m)

        # Nota especial para N estimado
        es_n_estimado = col_data == 'n_estimado_r'

        def estilo(feature):
            val = feature['properties'].get(col_wpavg)
            color = colormap(val) if val is not None else '#CCCCCC'
            return {'fillColor': color, 'color': '#333', 'weight': 1.2, 'fillOpacity': 0.72}

        # Tooltip personalizado
        nota_n = " (estimado: MO÷20)" if es_n_estimado else ""
        folium.GeoJson(
            gdf_plot.__geo_interface__,
            style_function=estilo,
            highlight_function=lambda x: {'weight': 3, 'fillOpacity': 0.95},
            tooltip=GeoJsonTooltip(
                fields=['musym', col_wpavg],
                aliases=['Unidad:', f'{label_sel}{nota_n}:'],
                localize=True,
                sticky=True,
                style="font-family:sans-serif; font-size:13px;"
            ),
            name=label_sel
        ).add_to(m)

        # Contorno del lote
        folium.GeoJson(
            gdf_lote.__geo_interface__,
            style_function=lambda x: {'color': 'white', 'weight': 2.5, 'fillOpacity': 0},
            name='Límite del lote'
        ).add_to(m)

        folium.LayerControl().add_to(m)

        # Título del mapa
        titulo_html = f"""
        <div style="position:fixed; top:10px; left:50%; transform:translateX(-50%);
                    background:white; padding:8px 18px; border-radius:8px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.25); font-family:sans-serif;
                    font-size:14px; font-weight:bold; z-index:9999; max-width:90%;">
            🗺️ {label_sel} &nbsp;·&nbsp; Promedio Ponderado 0–{PROF_MAX_CM} cm
            {'&nbsp;<span style="color:#e67e22;font-weight:bold;">⚠️ Estimado desde MO</span>' if es_n_estimado else ''}
        </div>
        """
        m.get_root().html.add_child(folium.Element(titulo_html))

        display(m)

        # Tabla resumen
        resumen_html = gdf_plot[['musym', col_wpavg]].rename(
            columns={'musym': 'Unidad', col_wpavg: f'{label_sel} (prom. ponderado 0–{PROF_MAX_CM} cm)'}
        ).style.format({f'{label_sel} (prom. ponderado 0–{PROF_MAX_CM} cm)': '{:.2f}'}).set_table_styles(
            [{'selector': 'th', 'props': [('background', '#2c3e50'), ('color', 'white'), ('padding', '8px 14px')]},
             {'selector': 'td', 'props': [('padding', '6px 14px'), ('text-align', 'center')]}]
        ).to_html()
        display(HTML(f"<div style='margin-top:12px;'>{resumen_html}</div>"))

btn_generar.on_click(generar_mapa)

panel_controles = widgets.VBox([
    widgets.HTML("<h3 style='margin:8px 0; font-family:sans-serif;'>🗺️ Bloque 1 — Mapa Coropleta</h3>"
                 "<p style='font-family:sans-serif; color:#555; font-size:13px; margin:0 0 10px;'>Seleccioná la variable y generá el mapa coropleta por promedio ponderado 0–80 cm.</p>"),
    widgets.HBox([selector_var, btn_generar])
], layout=widgets.Layout(padding='14px', border='1px solid #bdc3c7', border_radius='8px', margin='8px 0'))

display(panel_controles)
display(out_mapa)
generar_mapa()  # Generar con variable por defecto al ejecutar la celda


Output()

---
## 📊 BLOQUE 2 — Perfiles en Profundidad
Visualización del perfil completo del componente principal de cada unidad cartográfica.  

- **Textura**: barras apiladas horizontales (Arcilla + Limo + Arena).  
- **Otras variables**: gráfico de área con el eje Y invertido (profundidad real de cada horizonte).  
- **Fósforo (P)**: panel de datos no disponibles.  
- **Nitrógeno (N)**: estimado desde MO, con indicación explícita.

In [10]:
# ============================================================
# CELDA 6 — BLOQUE 2: PERFILES EN PROFUNDIDAD (PLOTLY)
# ============================================================

VARIABLES_PERFIL = {
    'Textura (Arcilla / Limo / Arena)': ['claytotal_r', 'silttotal_r', 'sandtotal_r'],
    'pH':                    'ph1to1h2o_r',
    'CE — Salinidad (dS/m)': 'ec_r',
    'PSI — Sodicidad (%)':   'esp_r',
    'SAR':                   'sar_r',
    'Materia Orgánica (%)':  'om_r',
    'CIC (cmol/kg)':         'cec7_r',
    'N estimado (%) ⚠️':     'n_estimado_r',
    '⚠️ Fósforo (P) — No disponible en SDA': None,
}

COLORES_TEXTURA = {
    'claytotal_r': '#8B0000',
    'silttotal_r': '#C68642',
    'sandtotal_r': '#F4A460',
}
NOMBRES_TEXTURA = {
    'claytotal_r': 'Arcilla',
    'silttotal_r': 'Limo',
    'sandtotal_r': 'Arena',
}

# ── Obtener lista de unidades disponibles ──
mukeys_disp = df_horiz['mukey'].unique().tolist()
musym_map   = gdf_clipped[['mukey','musym']].drop_duplicates().set_index('mukey')['musym'].to_dict()

opciones_unidades = [
    f"Unidad {musym_map.get(str(mk), mk)} (mukey: {mk})"
    for mk in mukeys_disp
]

selector_var_p = widgets.Dropdown(
    options=list(VARIABLES_PERFIL.keys()),
    value='Textura (Arcilla / Limo / Arena)',
    description='Variable:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='360px')
)

selector_unidad = widgets.SelectMultiple(
    options=opciones_unidades,
    value=opciones_unidades[:min(3, len(opciones_unidades))],
    description='Unidades:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='360px', height='100px')
)

btn_perfil = widgets.Button(
    description='📊 Generar Perfiles',
    button_style='success',
    layout=widgets.Layout(width='180px', margin='4px 8px')
)

out_perfiles = widgets.Output()

def generar_perfiles(b=None):
    with out_perfiles:
        out_perfiles.clear_output(wait=True)
        label_sel = selector_var_p.value
        col_data  = VARIABLES_PERFIL[label_sel]

        # ── Fósforo: dato no disponible ──
        if col_data is None:
            display(HTML("""
            <div style="background:#fff3cd; border:2px solid #ffc107; border-radius:10px;
                        padding:24px 30px; max-width:680px; font-family:sans-serif; margin:12px 0;">
                <h3 style="color:#856404; margin:0 0 10px;">⚠️ Fósforo (P) — Dato No Disponible en SDA</h3>
                <p style="color:#555; margin:0;">
                    La base de datos Soil Data Access (USDA-NRCS) no incluye datos de Fósforo
                    en la tabla <code>chorizon</code> ni en ninguna otra tabla tabular.
                    Para incluir este dato, podés agregar manualmente tu análisis de laboratorio.
                </p>
            </div>
            """))
            return

        # Filtrar unidades seleccionadas
        mukeys_sel = []
        for opt in selector_unidad.value:
            mk = opt.split('mukey: ')[1].rstrip(')')
            mukeys_sel.append(mk)

        if not mukeys_sel:
            print("⚠️ Seleccioná al menos una unidad.")
            return

        n_unidades = len(mukeys_sel)
        es_textura    = isinstance(col_data, list)
        es_n_estimado = col_data == 'n_estimado_r'

        fig = make_subplots(
            rows=1, cols=n_unidades,
            subplot_titles=[
                f"Unidad {musym_map.get(mk, mk)}"
                for mk in mukeys_sel
            ],
            shared_yaxes=True
        )

        for col_idx, mk in enumerate(mukeys_sel, start=1):
            df_mu = df_horiz[df_horiz['mukey'].astype(str) == str(mk)]
            if df_mu.empty:
                continue

            # Componente dominante
            comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
            df_comp  = df_mu[df_mu['cokey'] == comp_dom].sort_values('hzdept_r')
            compname = df_comp['compname'].iloc[0]
            comppct  = df_comp['comppct_r'].iloc[0]

            if es_textura:
                # ── Barras horizontales apiladas por fracción textural ──
                for var_tex in col_data:
                    if var_tex not in df_comp.columns:
                        continue
                    valores_x = []
                    valores_y = []
                    nombres_hz = []
                    for _, hz in df_comp.iterrows():
                        top = hz['hzdept_r']
                        bot = hz['hzdepb_r']
                        val = hz[var_tex]
                        mid = (top + bot) / 2
                        esp = bot - top
                        valores_x.append(val if not pd.isna(val) else 0)
                        valores_y.append(mid)
                        nombres_hz.append(f"{hz['hzname']} ({top}–{bot} cm)")

                    show_legend = (col_idx == 1)
                    fig.add_trace(
                        go.Bar(
                            x=valores_x,
                            y=valores_y,
                            orientation='h',
                            name=NOMBRES_TEXTURA[var_tex],
                            marker_color=COLORES_TEXTURA[var_tex],
                            text=[f"{v:.0f}%" if v > 0 else '' for v in valores_x],
                            textposition='inside',
                            hovertemplate=f"<b>{NOMBRES_TEXTURA[var_tex]}</b><br>Valor: %{{x:.1f}}%<br>Profundidad: %{{y}} cm<extra></extra>",
                            legendgroup=NOMBRES_TEXTURA[var_tex],
                            showlegend=show_legend,
                            width=[hz['hzdepb_r'] - hz['hzdept_r'] for _, hz in df_comp.iterrows()]
                        ),
                        row=1, col=col_idx
                    )

            else:
                # ── Gráfico de área para variable escalar ──
                if col_data not in df_comp.columns:
                    continue

                # Construir línea de perfil: cada horizonte como escalón
                xs, ys = [], []
                for _, hz in df_comp.iterrows():
                    val = hz[col_data]
                    if pd.isna(val): val = None
                    xs.extend([val, val])
                    ys.extend([hz['hzdept_r'], hz['hzdepb_r']])

                color_linea = '#e67e22' if es_n_estimado else '#2980b9'
                color_fill  = 'rgba(230,126,34,0.2)' if es_n_estimado else 'rgba(41,128,185,0.15)'

                fig.add_trace(
                    go.Scatter(
                        x=xs,
                        y=ys,
                        mode='lines',
                        fill='tozerox',
                        fillcolor=color_fill,
                        line=dict(color=color_linea, width=2.5),
                        name=compname,
                        showlegend=(col_idx == 1),
                        hovertemplate=f"<b>{label_sel}</b><br>Valor: %{{x:.2f}}<br>Prof.: %{{y}} cm<extra></extra>"
                    ),
                    row=1, col=col_idx
                )

                # Línea de referencia 80 cm (zona de raíces)
                fig.add_hline(
                    y=PROF_MAX_CM, line_dash='dash', line_color='#27ae60',
                    annotation_text=f'{PROF_MAX_CM} cm (zona raíz)',
                    annotation_position='bottom right',
                    row=1, col=col_idx
                )

            # Subtítulo con componente dominante
            fig.layout.annotations[col_idx - 1]['text'] = (
                f"<b>Unidad {musym_map.get(mk, mk)}</b><br>"
                f"<span style='font-size:11px;color:#555;'>{compname} ({comppct:.0f}%)</span>"
            )

        # ── Layout general ──
        barmode = 'stack' if es_textura else None
        prof_max_plot = df_horiz['hzdepb_r'].max() if not df_horiz.empty else 200

        nota_title = " — <span style='color:#e67e22;'>⚠️ Estimado: N = MO ÷ 20</span>" if es_n_estimado else ""

        fig.update_layout(
            title=dict(
                text=f"📊 Perfil en Profundidad — {label_sel}{nota_title}",
                font=dict(size=16)
            ),
            barmode=barmode,
            height=560,
            template='plotly_white',
            legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'),
            margin=dict(t=100, b=80)
        )

        # Eje Y invertido en todos los subplots (profundidad hacia abajo)
        for i in range(1, n_unidades + 1):
            yax = f'yaxis{i}' if i > 1 else 'yaxis'
            fig.update_layout(**{
                yax: dict(
                    autorange='reversed',
                    title='Profundidad (cm)' if i == 1 else '',
                    range=[prof_max_plot + 5, 0],
                    gridcolor='#eee'
                )
            })
            xax = f'xaxis{i}' if i > 1 else 'xaxis'
            x_title = '(%)' if es_textura else label_sel
            fig.update_layout(**{xax: dict(title=x_title, gridcolor='#eee')})

        fig.show()

        # Advertencia N estimado
        if es_n_estimado:
            display(HTML("""
            <div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px;
                        padding:12px 18px; font-family:sans-serif; font-size:13px; margin-top:8px; max-width:700px;">
                <b>⚠️ Nitrógeno estimado</b> — Los valores mostrados son una <em>aproximación teórica</em>
                calculada como <code>N = MO ÷ 20</code>, basada en la relación C:N de Bremner (1965).
                No reemplaza al análisis de laboratorio. La mineralización real depende de temperatura,
                humedad y actividad biológica.
            </div>
            """))

btn_perfil.on_click(generar_perfiles)

panel_perfiles = widgets.VBox([
    widgets.HTML("<h3 style='margin:8px 0; font-family:sans-serif;'>📊 Bloque 2 — Perfiles en Profundidad</h3>"
                 "<p style='font-family:sans-serif; color:#555; font-size:13px; margin:0 0 10px;'>Seleccioná variable y unidades para visualizar el perfil completo.</p>"),
    widgets.HBox([selector_var_p, selector_unidad, btn_perfil])
], layout=widgets.Layout(padding='14px', border='1px solid #bdc3c7', border_radius='8px', margin='8px 0'))

display(panel_perfiles)
display(out_perfiles)
generar_perfiles()  # Generar con valores por defecto al ejecutar la celda

Output()

---
## 📌 Notas finales

### Fósforo (P)
El dato de P **no existe** en ninguna tabla de la base de datos SDA (ni en `chorizon`, ni en `component`, ni en tablas interpretativas).  
El motivo es que la disponibilidad de P depende del historial de fertilización, pH, y mineralogía específica, variables que no se generalizan en modelos pedológicos nacionales.

### Nitrógeno (N) — Estimación
El **Nitrógeno Orgánico Total estimado** se calcula como:
$$N_{\text{estimado}} = \frac{MO}{20}$$
basado en la relación C/N promedio de la materia orgánica del suelo (Bremner, 1965).  
Este valor es una aproximación teórica y no reemplaza el análisis de laboratorio.

### Fuentes
- [USDA-NRCS Soil Data Access](https://sdmdataaccess.nrcs.usda.gov/)
- Bremner, J.M. (1965). Inorganic forms of nitrogen. *Methods of Soil Analysis*, Part 2.
- [SDA Column Descriptions](https://sdmdataaccess.nrcs.usda.gov/documents/TableRelationships.pdf)